# v53 train — Evidence Fusion Field(EV23 → DecoderC, 5fold)

全証拠(PF事後場12ch+v50系10ch+v50B)の (時間×δ) 場を単一デコーダで読む。
基準: v50B combo fold0=5.771 / 全体7.778。3本等重み6.757。目標=単体6.x。
バンクdropout(ch11-22, p=0.15)で混合過学習に対抗。fold=v50と同一(sorted[f::5])。


In [ ]:
# ===== 設定 + 証拠ロード =====
import os, json, math, time, gc, pickle
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
P0 = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
ART = P0 / "v53" / "artifacts_v53"; (ART / "nn53").mkdir(parents=True, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cuda":
    torch.cuda.set_per_process_memory_fraction(float(os.environ.get("VFRAC", "0.5")))
CH = int(os.environ.get("CH", "128")); DILT = (1, 2, 4, 8, 16)
EPOCHS = int(os.environ.get("EPOCHS", "60")); LR = 3e-4; LAM = 0.02
TOKMAX = int(os.environ.get("TOKMAX", "1200")); BW = 6
BDROP = float(os.environ.get("BDROP", "0.15"))     # バンクdropout(ch10-21 の12場)
NOISE = 0.02
FOLDS = [int(x) for x in os.environ.get("FOLDS", "0,1,2,3,4").split(",")]
ND = 181; NB = 90; STEP = 0.5
EVD = pickle.loads((ART / "evidenceC_v53.pkl").read_bytes())
WELLS = sorted(EVD.keys())
CIN = int(EVD[WELLS[0]]["EV"].shape[0])            # 23(tw/nbr) / 29(self込み)を自動追従
NPF_CH = CIN - 11                                   # PF事後場チャネル数(バンクdropout対象 = ch10..10+NPF_CH)
print(f"evidence: {len(WELLS)} wells  ch={CIN}  (PF場 {NPF_CH}ch)")


In [ ]:
# ===== モデル(DecoderC)+ バッチ =====
class DecoderC(nn.Module):
    def __init__(self, cin=CIN, ch=CH, dils=DILT):
        super().__init__()
        self.inp = nn.Conv2d(cin, ch, (5, 5), padding=(2, 2))
        self.blocks = nn.ModuleList(); self.norms = nn.ModuleList()
        for dt in dils:
            self.blocks.append(nn.Sequential(
                nn.Conv2d(ch, ch, (5, 5), padding=(2 * dt, 2), dilation=(dt, 1)), nn.GELU(),
                nn.Conv2d(ch, ch, 1)))
            self.norms.append(nn.GroupNorm(8, ch))
        self.gru = nn.GRU(ch, ch, batch_first=True, bidirectional=True)
        self.mix = nn.Linear(2 * ch, ch)
        self.out = nn.Conv2d(ch, 1, 1)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)
        self.ref = nn.Linear(2 * ch, 1)
        nn.init.zeros_(self.ref.weight); nn.init.zeros_(self.ref.bias)
    def forward(self, x):
        h = self.inp(x)
        for blk, nm in zip(self.blocks, self.norms):
            h = nm(h + blk(h))
        g = h.mean(3).transpose(1, 2)
        g2, _ = self.gru(g)
        h = h + self.mix(g2).transpose(1, 2)[:, :, :, None]
        return self.out(h)[:, 0], 2.0 * torch.tanh(self.ref(g2)[:, :, 0])

def make_batch(wns):
    Tm = max(EVD[w]["EV"].shape[1] for w in wns); B = len(wns)
    x = torch.zeros(B, CIN, Tm, ND); m = torch.zeros(B, Tm)
    dt = torch.full((B, Tm), float("nan")); wg = torch.zeros(B, Tm)
    for i, w in enumerate(wns):
        e = EVD[w]; T = e["EV"].shape[1]
        x[i, :, :T] = torch.from_numpy(e["EV"].astype(np.float32))
        m[i, :T] = 1.0
        dt[i, :T] = torch.from_numpy(e["dtrue"]); wg[i, :T] = torch.from_numpy(e["wgt"])
    return x.to(DEV), m.to(DEV), dt.to(DEV), wg.to(DEV)
offs_t = torch.arange(-NB, NB + 1, device=DEV).float() * STEP


In [ ]:
# ===== 学習(fold別, resume可)+ 評価 =====
def train_fold(f):
    val = WELLS[f::5]; trn = [w for w in WELLS if w not in set(val)]
    fp = ART / "nn53" / f"decC_f{f}.pt"
    net = DecoderC().to(DEV)
    if fp.exists():
        net.load_state_dict(torch.load(fp, map_location=DEV)); print(f"[f{f}] load")
        return net, val
    torch.manual_seed(f); rng = np.random.default_rng(f)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    lens = {w: EVD[w]["EV"].shape[1] for w in trn}
    def make_batches(order):
        ws_ = [trn[j] for j in order]
        wl = [w for i in range(0, len(ws_), 64) for w in sorted(ws_[i:i + 64], key=lambda x: lens[x])]
        batches = []; cur = []
        for w in wl:
            t = cur + [w]; Tmx = max(lens[x] for x in t)
            if cur and (len(t) * Tmx > TOKMAX or len(t) > BW):
                batches.append(cur); cur = [w]
            else:
                cur = t
        if cur: batches.append(cur)
        return batches
    t0 = time.time()
    for ep in range(EPOCHS):
        for pg in opt.param_groups:
            pg["lr"] = LR * 0.5 * (1 + math.cos(math.pi * ep / max(EPOCHS - 1, 1)))
        net.train(); losses = []; nskip = 0
        for wns in make_batches(rng.permutation(len(trn))):
            try:
                x, m, dtr, wg = make_batch(wns)
                if BDROP > 0:                              # バンクdropout(PF事後場チャネル)
                    msk = (torch.rand(x.shape[0], NPF_CH, device=DEV) > BDROP).float()[:, :, None, None]
                    x[:, 10:10 + NPF_CH] = x[:, 10:10 + NPF_CH] * msk
                if NOISE: x = x + NOISE * torch.randn_like(x)
                logits, refv = net(x)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); nskip += 1; continue
            kt = (dtr / STEP + NB)
            ok = m.bool() & torch.isfinite(dtr) & (kt >= 0) & (kt <= ND - 1) & (wg > 0)
            if not ok.any(): continue
            ktc = torch.nan_to_num(kt, nan=float(NB)).clamp(0, ND - 1)
            d = (torch.arange(ND, device=DEV)[None, None, :] - ktc[:, :, None])
            tgt = torch.exp(-0.5 * (d / 2.0) ** 2); tgt = tgt / tgt.sum(-1, keepdim=True).clamp_min(1e-12)
            lsm = F.log_softmax(logits, dim=-1)
            ce = -(tgt * lsm).sum(-1)
            P = lsm.exp(); dmean = (P * offs_t[None, None, :]).sum(-1) + refv
            se = (dmean - torch.nan_to_num(dtr)) ** 2
            wgn = wg / wg[ok].mean()
            loss = (ce[ok] * wgn[ok]).mean() + LAM * (se[ok] * wgn[ok]).mean()
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); nskip += 1; continue
            opt.zero_grad(set_to_none=True); loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            if not torch.isfinite(gn):
                opt.zero_grad(set_to_none=True); nskip += 1; continue
            opt.step(); losses.append(float(loss.detach()))
        if (ep + 1) % 5 == 0 or ep < 2:
            print(f"  [f{f}] ep{ep+1}/{EPOCHS} loss={np.mean(losses):.4f} skip={nskip} ({time.time()-t0:.0f}s)", flush=True)
    torch.save(net.state_dict(), fp)
    return net, val

@torch.no_grad()
def eval_fold(net, val, keep_extra=False):
    net.eval(); errs = []; extra = {}
    SH = np.arange(-20.0, 20.01, 0.5); MW = 12
    for w in val:
        e = EVD[w]
        x, m, _, _ = make_batch([w])
        lg, rf = net(x)
        T = e["EV"].shape[1]
        P = torch.softmax(lg[0, :T], dim=-1)
        dh = (P * offs_t[None, :]).sum(-1) + rf[0, :T]
        var = (P * (offs_t[None, :] - dh[:, None]) ** 2).sum(-1)
        dhn = dh.cpu().numpy(); z = e["st"] + dhn[e["bid"]]
        err = z - e["y_ev"]; errs.append(err[np.isfinite(err)])
        if keep_extra:
            Pn = P.cpu().numpy().astype(np.float64)
            stdn = np.sqrt(np.maximum(var.cpu().numpy(), 0.0))
            maxp = Pn.max(1)
            nb_ = T
            csum = np.concatenate([np.zeros((nb_, 1)), np.cumsum(Pn, 1)], 1)
            kk = np.round((dhn[:, None] + SH[None, :]) / STEP).astype(np.int64) + NB
            lo = np.clip(kk - MW, 0, ND); hi = np.clip(kk + MW + 1, 0, ND)
            mass = np.take_along_axis(csum, hi, 1) - np.take_along_axis(csum, lo, 1)
            prof = np.log(np.maximum(mass, 1e-9)).mean(0)
            marg = float(prof[np.abs(SH) <= 1.0].max() - prof[np.abs(SH) >= 4.0].max())
            extra[w] = (z.astype(np.float64), stdn[e["bid"]].astype(np.float32), marg,
                        maxp[e["bid"]].astype(np.float32))
    return errs, extra

agg = []; per_fold = {}; EXTRA = {}; E53 = {}
for f in FOLDS:
    net, val = train_fold(f)
    errs, extra = eval_fold(net, val, keep_extra=True)
    EXTRA.update(extra)
    for w, er in zip(val, errs): E53[w] = er.astype(np.float32)
    r = float(np.sqrt(np.mean(np.concatenate(errs) ** 2)))
    per_fold[f] = r; agg.extend(errs)
    print(f"[foldC{f}] v53 RMSE = {r:.4f}", flush=True)
    del net; gc.collect(); torch.cuda.empty_cache()
ee = np.concatenate(agg)
total = float(np.sqrt(np.mean(ee ** 2)))
print(f"\n=== v53 Evidence Fusion Field folds={FOLDS} プールRMSE = {total:.4f} ===")
print("  参考: v50B 7.778 / 3本等重み 6.757 / v52 GBM 6.27 / oracle(枝) 5.17")
np.savez_compressed(ART / "v53_errs.npz", **E53)
json.dump(dict(per_fold=per_fold, total=total), open(ART / "v53_results.json", "w"), indent=1)


In [ ]:
# ===== v53特徴を v52 parquet 行に整列して保存(スタック統合用)=====
import pyarrow.parquet as pq
import pandas as pd
PARQUET = P0 / "v52" / "artifacts_v52" / "data" / "train.parquet"
tb = pq.read_table(PARQUET, columns=["id", "well", "last_known_tvt"])
ids = tb.column("id").to_pandas().astype(str)
wname = tb.column("well").to_pandas().astype(str).to_numpy()
lk = tb.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
rn = ids.str.rsplit("_", n=1, expand=True)[1].astype(int).to_numpy()
N = len(ids); F4 = np.full((N, 4), np.nan, np.float32); miss = 0
for wn in pd.unique(wname):
    ii = np.where(wname == wn)[0]
    o = np.argsort(rn[ii], kind="stable"); iis = ii[o]
    dx = EXTRA.get(wn)
    if dx is None: miss += 1; continue
    z, sd, mg, mp = dx
    n0 = min(len(iis), len(z))
    F4[iis[:n0], 0] = (z[:n0] - lk[iis[:n0]]).astype(np.float32)
    F4[iis[:n0], 1] = sd[:n0]; F4[iis[:n0], 2] = mg; F4[iis[:n0], 3] = mp[:n0]
np.save(ART / "v53_feature.npy", F4)
print(f"saved v53_feature.npy [{N},4] miss={miss}  (cols: v53_delta/v53_std/v53_margin/v53_maxp)")
